In [ ]:
pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib


In [ ]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import csv

# Gmail read-only permission
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']


def gmail_authenticate():
    flow = InstalledAppFlow.from_client_secrets_file(
        'credentials.json', SCOPES)
    creds = flow.run_local_server(port=0)
    service = build('gmail', 'v1', credentials=creds)
    return service


def get_sent_emails(service, after_date, before_date):

    # Gmail search query
    query = f'in:sent after:{after_date} before:{before_date}'

    results = service.users().messages().list(
        userId='me',
        q=query
    ).execute()

    messages = results.get('messages', [])

    email_data = []   # store rows for CSV

    for msg in messages:
        msg_id = msg['id']

        message = service.users().messages().get(
            userId='me',
            id=msg_id,
            format='metadata',
            metadataHeaders=['Subject', 'To', 'Date']
        ).execute()

        headers = message['payload']['headers']

        subject = ""
        to_email = ""
        date = ""

        for header in headers:
            if header['name'] == 'Subject':
                subject = header['value']
            if header['name'] == 'To':
                to_email = header['value']
            if header['name'] == 'Date':
                date = header['value']

        # PRINT (existing behavior)
        print("Date:", date)
        print("To:", to_email)
        print("Subject:", subject)
        print("="*50)

        # SAVE FOR CSV
        email_data.append([date, to_email, subject])

    # WRITE CSV FILE
    with open('sent_emails.csv', mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)

        # header row
        writer.writerow(['Date', 'To Email', 'Subject'])

        # data rows
        writer.writerows(email_data)

    print("✅ Data saved to sent_emails.csv")


if __name__ == "__main__":

    service = gmail_authenticate()

    # YYYY/MM/DD format
    after_date = "2025/01/01"
    before_date = "2025/12/31"

    get_sent_emails(service, after_date, before_date)
